In [ ]:
%pip install xgboost lightgbm

In [ ]:
%run ../globalvariables

In [ ]:
%run ../core/base_model

In [ ]:
%run ../core/models

In [ ]:
%run ../core/evaluation

In [ ]:
%run ../core/tuning

In [ ]:
%run ../config/logger_config

In [ ]:
import yaml

In [ ]:
# Target gas widget
dbutils.widgets.dropdown("magnitud_target", "no2", [
    "no2", "no", "nox", "pm10", "pm2_5", "o3", "so2", "co",
    "tol", "ben", "ebe", "ch4", "nmhc", "tch",
])
MAGNITUD_TARGET = dbutils.widgets.get("magnitud_target")

MAGNITUD_MAP = {
    "no2": "NO2", "no": "NO", "nox": "NOx", "pm10": "PM10", "pm2_5": "PM2.5",
    "o3": "O3", "so2": "SO2", "co": "CO", "tol": "TOL", "ben": "BEN",
    "ebe": "EBE", "ch4": "CH4", "nmhc": "NMHC", "tch": "TCH",
}
MAGNITUD_VALUE = MAGNITUD_MAP[MAGNITUD_TARGET]
NOTEBOOK = "ml/domains/forecast_gas"
errors = []

In [ ]:
# Traffic columns pivoted wide
TRAFFIC_COLS = [
    f"{franja}_{metric}"
    for franja in ("Manana", "Tarde", "Noche")
    for metric in ("intensidad_media", "ocupacion_media", "carga_pico", "vmed_media")
]
TARGET_COL = "valor_medio"
FEATURE_COLS = [
    "n_estaciones", "es_festivo", "num_fabricas_distrito",
    "lag_1", "lag_7", "rolling_mean_7d", "rolling_mean_30d",
] + TRAFFIC_COLS

In [ ]:
# Load features and lags
gas = spark.table(f"{ML_TABLE}.features_{MAGNITUD_TARGET}")
gas = add_lag_features(gas, TARGET_COL, partition_cols=("cod_dis",)).toPandas()

# toPandas gives datetime.date
gas["fecha"] = pd.to_datetime(gas["fecha"])

train_pdf, test_pdf = temporal_split(gas)
X_train, y_train = train_pdf[FEATURE_COLS], train_pdf[TARGET_COL]
X_test, y_test = test_pdf[FEATURE_COLS], test_pdf[TARGET_COL]
logger.info(f"{MAGNITUD_VALUE}: {len(train_pdf)} train rows, {len(test_pdf)} test rows")

In [ ]:
from datetime import date

# One experiment folder per gas and date
mlflow.set_registry_uri("databricks-uc")
EXPERIMENT_PATH = f"{ML_EXPERIMENT_BASE}/{MAGNITUD_VALUE}_{date.today().isoformat()}"
mlflow.set_experiment(EXPERIMENT_PATH)
logger.info(f"experiment: {EXPERIMENT_PATH}")

# Shared hyperparameter ranges
with open("../config/model_search_space_aire.yml") as f:
    search_config = yaml.safe_load(f)

In [ ]:
# One flat run per model
best_per_model = []

for model_type, cfg in search_config["models"].items():
    try:
        model_class = MODEL_REGISTRY[model_type]
        param_grid = model_class.get_search_space(cfg.get("search_space"), cfg.get("n_grid_points"))
        run_id, mae, rmse, smape, params = run_gridsearch(
            model_class, model_type, param_grid,
            X_train, y_train, X_test, y_test, log_model=False,
            run_name=f"{MAGNITUD_VALUE}_{model_type}",
            n_iter=cfg.get("n_iter"), cv_splits=cfg.get("cv_splits", 5),
        )
        best_per_model.append((model_type, run_id, mae, rmse, smape, params))
        logger.info(f"{model_type}: mae={mae:.3f} rmse={rmse:.3f} smape={smape:.3f} params={params}")
    except Exception as e:
        errors.append(error_record(NOTEBOOK, e))
        logger.error(f"fail {model_type}: {type(e).__name__}: {e}")

In [ ]:
# Pick and refit the winner only
best_model_type, best_run_id, best_mae, best_rmse, best_smape, best_params = min(best_per_model, key=lambda t: t[2])
model = MODEL_REGISTRY[best_model_type].build(best_params)
model.fit(X_train, y_train)
logger.info(f"winner: {best_model_type} mae={best_mae:.3f} rmse={best_rmse:.3f} smape={best_smape:.3f}")

In [ ]:
# Holidays known in advance
calendario = (
    spark.table(f"{GOLD_TABLE}.dim_fecha")
    .select("fecha", "es_festivo")
    .toPandas()
    .set_index("fecha")["es_festivo"]
)
calendario.index = pd.to_datetime(calendario.index)

history = gas.set_index(["cod_dis", "fecha"])[TARGET_COL]

# Last known row per district
seeds = gas.sort_values("fecha").groupby("cod_dis").tail(1).set_index("cod_dis")

# Districts whose data has gone stale get skipped, not forecast from a
# months-old anchor date
STALE_CUTOFF_DAYS = 14
global_max_fecha = gas["fecha"].max()
stale_mask = seeds["fecha"] < (global_max_fecha - pd.Timedelta(days=STALE_CUTOFF_DAYS))
if stale_mask.any():
    logger.warning(
        f"skipping stale districts (no data in last {STALE_CUTOFF_DAYS}d): "
        f"{dict(seeds.loc[stale_mask, 'fecha'])}"
    )
seeds = seeds.loc[~stale_mask]

In [ ]:
# Same month, same weekday
SIMILAR_COLS = ["rolling_mean_7d", "rolling_mean_30d"] + TRAFFIC_COLS
similar_avg = (
    gas.assign(mes=gas["fecha"].dt.month, dia_semana=gas["fecha"].dt.dayofweek)
    .groupby(["cod_dis", "mes", "dia_semana"])[SIMILAR_COLS]
    .mean()
)
logger.info(f"similar day table: {len(similar_avg)} combos")

In [ ]:
# Recursive 7 day forecast
def recursive_forecast(fit_model):
    predicted = {}
    pred_rows = []

    for cod_dis, seed in seeds.iterrows():
        last_fecha = seed["fecha"]
        for h in range(1, FORECAST_HORIZON_GAS_DAYS + 1):
            target_fecha = last_fecha + pd.Timedelta(days=h)
            lag1_fecha = target_fecha - pd.Timedelta(days=1)
            lag7_fecha = target_fecha - pd.Timedelta(days=7)

            lag_1 = predicted.get((cod_dis, lag1_fecha), history.get((cod_dis, lag1_fecha)))
            lag_7 = history.get((cod_dis, lag7_fecha))

            # Seed as fallback
            key = (cod_dis, target_fecha.month, target_fecha.dayofweek)
            similar = similar_avg.loc[key] if key in similar_avg.index else seed

            row = {
                "n_estaciones": seed["n_estaciones"],
                "es_festivo": bool(calendario.get(target_fecha, False)),
                "num_fabricas_distrito": seed["num_fabricas_distrito"],
                "lag_1": lag_1,
                "lag_7": lag_7,
            }
            for col in SIMILAR_COLS:
                row[col] = similar[col]

            # Force numeric dtype: a missing lag (None) makes a 1-row
            # DataFrame infer dtype=object, which LightGBM rejects
            X_pred = pd.DataFrame([row])[FEATURE_COLS].astype(float)
            valor_predicho = float(fit_model.predict(X_pred)[0])
            predicted[(cod_dis, target_fecha)] = valor_predicho

            pred_rows.append({
                "distrito": cod_dis,
                "fecha": target_fecha,
                "valor_predicho": valor_predicho,
                "_h": h,
            })

    return pd.DataFrame(pred_rows)


pred_df = recursive_forecast(model)

In [ ]:
# Reopen the winning run to log the model, the forecast timeline, and mark it as best
with mlflow.start_run(run_id=best_run_id):
    mlflow.sklearn.log_model(
        model.estimator, artifact_path="model",
        input_example=X_train.head(3),
        signature=mlflow.models.infer_signature(X_train, y_train),
    )
    for h, group in pred_df.groupby("_h"):
        mlflow.log_metric(f"pred_mean_h{h}", group["valor_predicho"].mean())
    MlflowClient().set_tag(best_run_id, "mlflow.runName", f"{MAGNITUD_VALUE}_{best_model_type}_best")

pred_df = pred_df.drop(columns="_h")

# Promote only if better
version, promoted = promote_if_better(MAGNITUD_TARGET, best_run_id, best_mae)
logger.info(f"registered v{version}, promoted={promoted}")

# Log the same per-horizon diagnostic for every non-winning model, so
# pred_mean_h1..h7 is comparable across all model types, not just the winner
for model_type, run_id, mae, rmse, smape, params in best_per_model:
    if run_id == best_run_id:
        continue
    candidate_model = MODEL_REGISTRY[model_type].build(params)
    candidate_model.fit(X_train, y_train)
    candidate_pred_df = recursive_forecast(candidate_model)
    with mlflow.start_run(run_id=run_id):
        for h, group in candidate_pred_df.groupby("_h"):
            mlflow.log_metric(f"pred_mean_h{h}", group["valor_predicho"].mean())

In [ ]:
# One table per gas
pred_df["fecha"] = pred_df["fecha"].dt.date
write_ml(spark.createDataFrame(pred_df), f"forecast_{MAGNITUD_TARGET}_7", mode="overwrite")

log_errors(errors)
logger.info(f"forecast_{MAGNITUD_TARGET}_7 updated: {len(pred_df)} rows, run={best_run_id}")